# DxBench runner (Colab / Kaggle, T4)

Runtime: GPU (T4). Run cells top to bottom. Kaggle needs Internet enabled in the notebook settings.

In [ ]:
import os, subprocess

# ---- set these ----
REPO_URL = "https://github.com/vixrino/monarch-fft-mathdx.git"
BRANCH = "main"
CUDA_ARCH = "75"
# MathDx packages, checked on developer.nvidia.com/cufftdx-downloads (2026-11-17).
# 26.x ships for CUDA 13 only; the newest CUDA 12 build is 25.12.1.
MATHDX_URLS = {
    "13": "https://developer.nvidia.com/downloads/compute/cuFFTDx/redist/cuFFTDx/cuda13/nvidia-mathdx-26.10.0-cuda13.tar.gz",
    "12": "https://developer.nvidia.com/downloads/compute/cuFFTDx/redist/cuFFTDx/cuda12/nvidia-mathdx-25.12.1-cuda12.tar.gz",
}
# -------------------

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
cuda_major = nvcc.split("release ")[1].split(".")[0]
MATHDX_URL = MATHDX_URLS[cuda_major]
os.environ.update(WORK=WORK, REPO_URL=REPO_URL, BRANCH=BRANCH, CUDA_ARCH=CUDA_ARCH,
                  MATHDX_URL=MATHDX_URL, REPO=f"{WORK}/dxbench", DX_DIR=f"{WORK}/mathdx")
print(nvcc.strip().splitlines()[-1])
print("work dir:", WORK)
print("MathDx:", MATHDX_URL)

In [ ]:
%%bash
set -e
cmake --version | head -1
# the project and the cuFFTDx examples need CMake >= 3.26
python3 - <<'PY' || pip install -q "cmake>=3.26"
import subprocess, sys
v = subprocess.run(["cmake", "--version"], capture_output=True, text=True).stdout.split()[2]
sys.exit(0 if tuple(map(int, v.split(".")[:2])) >= (3, 26) else 1)
PY
hash -r; cmake --version | head -1

In [ ]:
%%bash
set -e
rm -rf "$REPO"
git clone -q --branch "$BRANCH" "$REPO_URL" "$REPO"
git -C "$REPO" log -1 --oneline

In [ ]:
%%bash
set -e
mkdir -p "$DX_DIR"
cd "$DX_DIR"
f=$(basename "$MATHDX_URL")
[ -f "$f" ] || curl -fL -o "$f" "$MATHDX_URL"
if ! tar -tzf "$f" > /dev/null 2>&1; then
  echo "download is not a tar.gz (license page or login wall?). First bytes:"; head -c 300 "$f"; exit 1
fi
tar -xzf "$f"
# archives unpack to nvidia-mathdx-<ver>/nvidia/mathdx/YY.MM/
ls -d "$DX_DIR"/*/nvidia/mathdx/*/

In [ ]:
import glob
roots = sorted(glob.glob(f"{os.environ['DX_DIR']}/*/nvidia/mathdx/*/"))
assert roots, "no nvidia/mathdx/YY.MM directory after unpacking"
os.environ["MATHDX_ROOT"] = roots[-1].rstrip("/")
print("MATHDX_ROOT =", os.environ["MATHDX_ROOT"])

In [ ]:
%%bash
cd "$REPO" && bash scripts/env_check.sh

In [ ]:
%%bash
set -e
cd "$REPO"
cmake -S . -B build -DCUDA_ARCH=$CUDA_ARCH -DMATHDX_ROOT="$MATHDX_ROOT"
cmake --build build -j
./build/dx_smoke

Smoke test: the official cuFFTDx `convolution_performance` example, built from the examples shipped in the package.

In [ ]:
%%bash
set -e
src=$(find "$MATHDX_ROOT" -name convolution_performance.cu | head -1)
[ -n "$src" ] || { echo "convolution_performance.cu not found in package"; exit 1; }
ex=$(dirname "$(dirname "$src")")
echo "examples dir: $ex"
cmake -S "$ex" -B "$WORK/build_cufftdx_examples" \
  -DCMAKE_BUILD_TYPE=Release \
  -Dmathdx_ROOT="$MATHDX_ROOT" \
  -DCUFFTDX_CUDA_ARCHITECTURES=${CUDA_ARCH}-real
# target name is <dir>_<file> in NVIDIA's examples CMakeLists; read it back to be sure
tgt=$(cmake --build "$WORK/build_cufftdx_examples" --target help | grep -o '[A-Za-z0-9_]*convolution_performance' | head -1)
echo "target: $tgt"
cmake --build "$WORK/build_cufftdx_examples" --target "$tgt" -j
bin=$(find "$WORK/build_cufftdx_examples" -type f -name convolution_performance -perm -u+x | head -1)
echo "running $bin"
"$bin"

## Phase 2: cuFFTDx baseline

Accuracy check (exit code != 0 on failure), then the timing sweep, written as JSON lines under `results/`.

In [ ]:
%%bash
set -e
cd "$REPO"
ctest --test-dir build --output-on-failure

In [ ]:
import datetime, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")[0]
gpu = gpu.strip().lower().replace(" ", "_")
RESULTS = f"{os.environ['REPO']}/results/{gpu}/{datetime.date.today().isoformat()}_cufftdx_block_fft.jsonl"
os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
os.environ["RESULTS"] = RESULTS
print(RESULTS)

In [ ]:
%%bash
set -e
cd "$REPO"
./build/cufftdx_block_fft --out "$RESULTS"
wc -l "$RESULTS"

In [ ]:
%%bash
cd "$REPO"
bash scripts/sass_check.sh build/cufftdx_block_fft block_fft

In [ ]:
# results are never edited by hand: download the file and commit it as is
try:
    from google.colab import files
    files.download(os.environ["RESULTS"])
except ImportError:
    print("Kaggle: file is at", os.environ["RESULTS"])